In [7]:
# Make sure GPU is enabled: Runtime → Change runtime type → GPU
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float())  # binarize
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)  # larger batch is OK on GPU

100%|██████████| 9.91M/9.91M [00:00<00:00, 120MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 35.6MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 33.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.56MB/s]


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class MemoryEfficientNADE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=500):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.W = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)
        self.V = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.01)
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))
        self.b_v = nn.Parameter(torch.zeros(input_dim))

    def forward(self, x):
        batch_size = x.size(0)
        h_cum = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        p = torch.zeros_like(x)

        for i in range(self.input_dim):
            h = torch.sigmoid(h_cum + self.b_h)
            p[:, i] = torch.sigmoid(h @ self.V[i].t() + self.b_v[i])
            h_cum += x[:, i].unsqueeze(1) * self.W[:, i].unsqueeze(0)

        return p

In [4]:
def train_nade(model, train_loader, epochs=50, lr=0.001):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        for x, _ in train_loader:
            x = x.view(x.size(0), -1).to(device)
            optimizer.zero_grad()
            p = model(x)
            loss = F.binary_cross_entropy(p, x)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

In [6]:
model = MemoryEfficientNADE(input_dim=784, hidden_dim=500)
train_nade(model, train_loader, epochs=20, lr=0.001)

Epoch 1/20, Avg Loss: 0.2080


KeyboardInterrupt: 

In [ ]:
def sample_nade(model, n_samples):
    model.to('cpu')  # sampling on CPU
    x = torch.zeros(n_samples, model.input_dim)
    h_cum = torch.zeros(n_samples, model.hidden_dim)

    for i in range(model.input_dim):
        h = torch.sigmoid(h_cum + model.b_h)
        p = torch.sigmoid(h @ model.V[i].t() + model.b_v[i])
        x[:, i] = torch.bernoulli(p)
        h_cum += x[:, i].unsqueeze(1) * model.W[:, i].unsqueeze(0)

    return x

In [ ]:
import matplotlib.pyplot as plt

samples = sample_nade(model, 16).view(-1, 1, 28, 28).detach()

plt.figure(figsize=(6,6))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(samples[i,0], cmap='gray')
    plt.axis('off')
plt.show()